# TrialMatch: clearer-prompt and class-balancing experiments

The original run reached 70.4% accuracy and 41.2% macro-F1, with no predictions for two
rare classes. We now test two separate recipes on the **same validation split**:

- **A:** prompt v2 with explicit label definitions and an exact JSON example; no class weights.
- **B:** the same prompt and training setup, with capped training-only class weights.

Both start from the original pinned **Qwen2.5-1.5B-Instruct** base, use three epochs and
exactly the same training-example order and optimizer-update budget. B reuses only A's
untouched-base validation results after verifying identical protocol and runtime provenance.
No previous trained adapter is used to initialize either run. The **test set stays unused**.

Choose **Runtime → Change runtime type → A100** and use the **newest** companion source ZIP.
Your previous full run was on a T4; this notebook checks the actual assigned GPU explicitly.
Improvement is not guaranteed: keep both runs, including any regressions.


## 1. Upload the updated companion ZIP

Upload `trialmatch-colab-starter.zip` from this experiment delivery. Older bundles are rejected.
The original experiment's source and results have been preserved separately.


In [ ]:
from pathlib import Path
import hashlib
import os
import stat
import zipfile
from google.colab import files

uploaded = files.upload()
expected = 'trialmatch-colab-starter.zip'
if expected not in uploaded:
    raise ValueError(f'Upload the updated {expected}.')
bundle_bytes = uploaded[expected]
BUNDLE_SHA256 = hashlib.sha256(bundle_bytes).hexdigest()
PROJECT = Path('/content/trialmatch-training-' + BUNDLE_SHA256[:12])
archive_path = Path('/content') / expected
archive_path.write_bytes(bundle_bytes)
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    if not {'src/trialmatch/prompts.py', 'configs/experiments/03_prompt_v2_balanced.toml', 'scripts/compare_ablations.py'} <= set(archive.namelist()):
        raise ValueError('This is an older bundle. Upload the updated ablation bundle.')
    for entry in archive.infolist():
        if not (PROJECT / entry.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Archive contains an unsafe path')
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError('Archive contains a symbolic link')
    archive.extractall(PROJECT)
del uploaded, bundle_bytes
os.chdir(PROJECT)
os.environ['HF_HOME'] = '/content/trialmatch-hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLCONFIGDIR'] = '/content/trialmatch-matplotlib'
print('Project:', PROJECT)
print('Bundle SHA256:', BUNDLE_SHA256)


## 2. Install dependencies and verify the actual GPU

`REQUIRE_A100 = True` stops before training if Colab assigns a different GPU. Select A100
in Runtime settings to use your allocation. Setting it to False permits another CUDA GPU
and records it accurately; the implementation itself does not require paid hardware.
Use the same runtime for A and B. Colab's CUDA PyTorch is preserved.


In [ ]:
import subprocess
import sys
import json

REQUIRE_A100 = True
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt', '-e', '.'], check=True)
hardware = json.loads(subprocess.check_output([sys.executable, '-c',
    'import torch,json; assert torch.cuda.is_available(), "Select a GPU runtime"; '
    'print(json.dumps({"gpu":torch.cuda.get_device_name(0),'
    '"memory_gib":torch.cuda.get_device_properties(0).total_memory/2**30,'
    '"torch":torch.__version__}))'], text=True))
print(hardware)
if REQUIRE_A100 and 'A100' not in hardware['gpu']:
    raise RuntimeError('This session is not using A100. Change the runtime GPU, or explicitly set REQUIRE_A100=False.')


## 3. Mount Drive

Both runs save completed-epoch optimizer checkpoints and adapters to Drive. These are
needed for resuming and are larger than the portable review ZIP. Keep the run directories.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
OUTPUT_ROOT = Path('/content/drive/MyDrive/TrialMatch/results')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 4. Recreate the pinned data split and run correctness checks

The data source and 37/8/8 patient split have not changed. Preprocessing mechanically
prepares all split files, but the development workflow opens only training and validation.
Tests use synthetic examples and tiny random models, never the held-out test patients.


In [ ]:
manifest = json.loads((PROJECT / 'data/manifests/split_manifest.json').read_text())
subprocess.run([sys.executable, 'scripts/prepare_data.py', '--revision', manifest['dataset_revision']], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## 5. Set shared settings and an experiment directory

Leave `RESUME_EXPERIMENT_ROOT` empty for a new A/B pair. To resume after disconnecting,
use the same uploaded bundle and settings, and paste the earlier **parent experiment path**
printed below. Each child run resumes from its last completed epoch; an incomplete epoch
is repeated. Do not use the original prompt-v1 run path here.

A/B differ only in their class weights. Every accepted training example appears once per
epoch in both runs. The rare-class relative weight is capped at 3, with counts computed
separately for inclusion/exclusion using training data only. No validation balancing.


In [ ]:
from datetime import datetime, timezone
import re

RESUME_EXPERIMENT_ROOT = ''  # Parent ablation-... folder, not A/B child or original run.
EPOCHS = 3
TRAIN_BATCH_SIZE = 2
ACCUMULATION_STEPS = 4
EVAL_BATCH_SIZE = 4
LEARNING_RATE = 0.0002

RUN_CONFIGS = {}
for name, source in [('A_prompt_v2', '02_prompt_v2.toml'),
                     ('B_prompt_v2_balanced', '03_prompt_v2_balanced.toml')]:
    settings = (PROJECT / 'configs/experiments' / source).read_text()
    for key, value in {'epochs':EPOCHS, 'per_device_batch_size':TRAIN_BATCH_SIZE,
                       'gradient_accumulation_steps':ACCUMULATION_STEPS,
                       'batch_size':EVAL_BATCH_SIZE, 'learning_rate':LEARNING_RATE}.items():
        settings, count = re.subn(r'(?m)^' + re.escape(key) + r'\s*=.*$', f'{key} = {value}', settings)
        if count != 1:
            raise ValueError(f'Unexpected configuration entry: {key}')
    destination = PROJECT / 'configs' / (name + '-run.toml')
    destination.write_text(settings)
    RUN_CONFIGS[name] = destination
plan = {'bundle_sha256':BUNDLE_SHA256,
        'configs':{name:path.read_text() for name,path in RUN_CONFIGS.items()},
        'test_set_used':False}
if RESUME_EXPERIMENT_ROOT:
    EXPERIMENT_ROOT = Path(RESUME_EXPERIMENT_ROOT)
    if not (EXPERIMENT_ROOT / 'experiment_plan.json').exists():
        raise ValueError('Resume path must be an existing A/B parent folder with experiment_plan.json')
    if json.loads((EXPERIMENT_ROOT / 'experiment_plan.json').read_text()) != plan:
        raise ValueError('Resume requires the same source bundle and settings')
else:
    EXPERIMENT_ROOT = OUTPUT_ROOT / ('ablation-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=False)
    (EXPERIMENT_ROOT / 'experiment_plan.json').write_text(json.dumps(plan, indent=2))
RUN_A = EXPERIMENT_ROOT / 'A_prompt_v2'
RUN_B = EXPERIMENT_ROOT / 'B_prompt_v2_balanced'
print('Keep this parent path for resuming:', EXPERIMENT_ROOT)


## 6. Run A, then B

A first measures the new untouched-base baseline, then trains unweighted adapters.
B verifies/reuses that same baseline and trains fresh adapters with loss weights. Both
use the same checkpoint selection rule and validate the reloaded selected adapter.
The source, data, model, prompt, generation and runtime checks prevent baseline reuse
across different protocols. A completed child is checked with `--resume` rather than retrained.


In [ ]:
def run_child(name, run_dir, baseline_from=None):
    run_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', 'scripts/run_training.py',
               '--output-dir', str(run_dir), '--config', str(RUN_CONFIGS[name])]
    if (run_dir / 'run.json').exists():
        command.append('--resume')
    if baseline_from is not None:
        command.extend(['--baseline-from', str(baseline_from)])
    with (run_dir / 'console.log').open('a') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f'{name} failed. Preserve {run_dir}/console.log and status/failure files.')

run_child('A_prompt_v2', RUN_A)
run_child('B_prompt_v2_balanced', RUN_B, baseline_from=RUN_A)


## 7. Compare measured results

The comparison script refuses to call this a controlled A/B comparison if settings,
model/data/runtime/source provenance, base metrics or update/example budgets differ.
Look at rare-class recall/F1, evidence precision/recall, invalid outputs and overall scores.
Each recipe selects its own epoch using the same rule. The original prompt-v1 T4 result
is historical context, not a same-hardware estimate of the prompt's isolated effect.


In [ ]:
from IPython.display import Markdown, display, Image

COMPARISON_DIR = EXPERIMENT_ROOT / 'comparison'
subprocess.run([sys.executable, 'scripts/compare_ablations.py',
                '--unweighted', str(RUN_A), '--balanced', str(RUN_B),
                '--output-dir', str(COMPARISON_DIR)], check=True)
display(Markdown((COMPARISON_DIR / 'ABLATION-COMPARISON.md').read_text()))
for title, run_dir in [('A: prompt v2', RUN_A), ('B: prompt v2 plus class weights', RUN_B)]:
    print(title)
    print((run_dir / 'status.json').read_text())
    display(Image(filename=str(run_dir / 'training-validation-curves.png')))
print('Actual class weights:')
print((RUN_B / 'class_weighting.json').read_text())


## 8. Export one review ZIP containing both runs

The combined ZIP contains each selected adapter plus logs, predictions, metrics and plots,
and the A/B comparison. Full optimizer files and nonselected adapters remain on Drive.
This portable review ZIP is not a resume archive. Share it for error analysis; keep test unused.


In [ ]:
import shutil

stage = Path('/content') / ('export-' + EXPERIMENT_ROOT.name)
stage.mkdir(parents=True, exist_ok=True)
for name, run_dir in [('A_prompt_v2', RUN_A), ('B_prompt_v2_balanced', RUN_B)]:
    child_zip = Path('/content') / (EXPERIMENT_ROOT.name + '-' + name + '.zip')
    subprocess.run([sys.executable, 'scripts/export_training_results.py', '--run-dir', str(run_dir),
                    '--output', str(child_zip)], check=True)
    with zipfile.ZipFile(child_zip) as archive:
        archive.extractall(stage / name)  # Archive was just produced by the local exporter.
shutil.copytree(COMPARISON_DIR, stage / 'comparison', dirs_exist_ok=True)
shutil.copyfile(EXPERIMENT_ROOT / 'experiment_plan.json', stage / 'experiment_plan.json')
result_zip = shutil.make_archive('/content/' + EXPERIMENT_ROOT.name + '-results', 'zip', root_dir=stage)
files.download(result_zip)


## Afterwards

Interpret the measured changes and failures before further tuning. A higher accuracy with
worse minority recall may be undesirable. Do not promise a score increase from balancing.
The next implementation work remains custom KV caching, tiled attention, speculative
decoding, the browser app and final held-out testing after configuration freeze.
